## 03 — Map Controls

So far we have added layers to a map by calling `m.add()`. Controls work the same way — they are widgets you attach to the map — but instead of drawing features, they add **UI elements**: layer toggles, a scale bar, a fullscreen button, a minimap.

This lesson covers the controls you will use most often.

## Layers vs Controls

| `m.add(...)` argument | What it does |
|---|---|
| `GeoJSON(...)` | draws geographic features on the map |
| `TileLayer(...)` | draws a background tileset |
| `LayersControl(...)` | adds a toggle panel for named layers |
| `ScaleControl(...)` | adds a distance scale bar |
| `FullscreenControl(...)` | adds a fullscreen button |
| `MiniMap(...)` | adds a small overview map in the corner |

All of these go through the same `m.add()` call. The map does not distinguish — it just manages a list of things attached to it.

## Setup

Load the GeoJSON data from the previous lesson and split it by geometry type.

In [1]:
import json
from ipyleaflet import Map, GeoJSON

WICHITA_FALLS = (33.9137, -98.4934)

with open("data/wichita_falls.geojson") as f:
    all_features = json.load(f)["features"]

points   = [f for f in all_features if f["geometry"]["type"] == "Point"]
lines    = [f for f in all_features if f["geometry"]["type"] == "LineString"]
polygons = [f for f in all_features if f["geometry"]["type"] == "Polygon"]

def make_fc(features):
    return {"type": "FeatureCollection", "features": features}

## LayersControl

`LayersControl` adds a panel in the corner that lets users toggle layers on and off. For it to work, each layer needs a **`name`** — that name appears as the label in the toggle panel.

In [2]:
from ipyleaflet import Map, GeoJSON, LayersControl

m = Map(center=WICHITA_FALLS, zoom=12)

# Give each layer a name — this is what appears in the toggle panel
point_layer = GeoJSON(data=make_fc(points),   name="Points of Interest")
line_layer  = GeoJSON(data=make_fc(lines),    name="Routes")
poly_layer  = GeoJSON(data=make_fc(polygons), name="Park Boundary")

m.add(point_layer)
m.add(line_layer)
m.add(poly_layer)

# Add the control — it automatically picks up all named layers
m.add(LayersControl())

m

Map(center=[33.9137, -98.4934], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'z…

Click the stacked-layers icon in the top-right to expand the panel. Each named layer has a checkbox — uncheck one to hide it.

Layers added **without** a `name` do not appear in the toggle panel.

## ScaleControl

`ScaleControl` draws a scale bar that updates as you zoom. It shows both metric and imperial units by default.

In [3]:
from ipyleaflet import Map, ScaleControl

m = Map(center=WICHITA_FALLS, zoom=12)
m.add(ScaleControl(position="bottomleft"))
m

Map(center=[33.9137, -98.4934], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'z…

The `position` argument accepts `"topleft"`, `"topright"`, `"bottomleft"`, or `"bottomright"`. All controls support this argument.

## FullscreenControl

`FullscreenControl` adds a button that expands the map to fill the browser window — useful when inspecting dense data.

In [5]:
from ipyleaflet import Map, GeoJSON, FullScreenControl 

m = Map(center=WICHITA_FALLS, zoom=12)
m.add(GeoJSON(data=make_fc(all_features)))
m.add(FullScreenControl())
m

Map(center=[33.9137, -98.4934], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'z…

## MiniMap

`MiniMap` places a small overview map in a corner. It stays zoomed out while you navigate the main map — useful for keeping track of where you are at high zoom levels.

In [14]:
from ipyleaflet import Map, WidgetControl

# 1. Create your main map
m = Map(center=(33.9137, -98.4934), zoom=12)

# 2. Create a smaller map to serve as the "MiniMap"
# We make it small (200x200) and remove the zoom controls so it looks like a UI element
minimap_widget = Map(
    center=m.center, 
    zoom=m.zoom - 5, 
    zoom_control=False, 
    attribution_control=False
)
minimap_widget.layout.width = '200px'
minimap_widget.layout.height = '150px'

# 3. Add the small map to the main map using WidgetControl
minimap_control = WidgetControl(widget=minimap_widget, position='bottomright')
m.add(minimap_control)

m

Map(center=[33.9137, -98.4934], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'z…

## Putting It Together

Controls compose — you can add as many as you need to a single map.

In [18]:
import json
from ipyleaflet import Map, GeoJSON, LayersControl, ScaleControl, FullScreenControl, WidgetControl

# 1. Initialize the Main Map
m = Map(center=WICHITA_FALLS, zoom=12)

# 2. Add GeoJSON Layers (Ensure make_fc and points/lines/polygons are defined)
m.add(GeoJSON(data=make_fc(points),   name="Points of Interest"))
m.add(GeoJSON(data=make_fc(lines),    name="Routes"))
m.add(GeoJSON(data=make_fc(polygons), name="Park Boundary"))

# 3. Add Working Controls
m.add(LayersControl(position="topright"))
m.add(ScaleControl(position="bottomleft"))
m.add(FullScreenControl(position="topleft")) # Fixed the capital 'S'

# 4. Create the "MiniMap" manually since your version lacks the shortcut
minimap_view = Map(
    center=m.center, 
    zoom=m.zoom - 5, 
    zoom_control=False, 
    attribution_control=False
)
minimap_view.layout.width = '150px'
minimap_view.layout.height = '150px'

# Wrap the small map in a WidgetControl to put it in the corner
m.add(WidgetControl(widget=minimap_view, position='bottomright'))

m

Map(center=[33.9137, -98.4934], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'z…

## Exercise A

Rebuild the "Putting It Together" map using `basemaps.Esri.WorldImagery` (satellite tiles) instead of the default street map. Keep all three named layers and all four controls.

In [20]:
import json
from ipyleaflet import Map, GeoJSON, LayersControl, ScaleControl, FullScreenControl, WidgetControl, basemaps

# 1. Use the satellite basemap
m = Map(center=WICHITA_FALLS, zoom=12, basemap=basemaps.Esri.WorldImagery)

# 2. Prepare layers
point_layer = GeoJSON(data=make_fc(points),   name="Points of Interest")
line_layer  = GeoJSON(data=make_fc(lines),    name="Routes")
poly_layer  = GeoJSON(data=make_fc(polygons), name="Park Boundary", visible=False)

# 3. Add layers to the map
m.add(point_layer)
m.add(line_layer)
m.add(poly_layer)

# 4. Add Controls using your specific version's names
m.add(LayersControl(position="topright"))
m.add(ScaleControl(position="bottomleft"))
m.add(FullScreenControl(position="topleft")) # Fixed the capital 'S'

# 5. Create the MiniMap manually since 'MiniMap' is missing from your library
minimap_view = Map(
    center=m.center, 
    zoom=m.zoom - 5, 
    zoom_control=False, 
    attribution_control=False
)
minimap_view.layout.width = '150px'
minimap_view.layout.height = '150px'

# We use WidgetControl (which was in your list!) to place the small map
m.add(WidgetControl(widget=minimap_view, position='bottomright'))

m

Map(center=[33.9137, -98.4934], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'z…

## Exercise B

ipyleaflet layers have a `visible` attribute you can set at any time — the map updates live without re-rendering.

1. Rebuild the composite map with the polygon layer starting **hidden** (`visible=False`)
2. Display the map — confirm the polygon layer is absent
3. In a new cell, set `poly_layer.visible = True` and watch the map update in place

In [21]:
from ipyleaflet import Map, GeoJSON, LayersControl, ScaleControl, FullScreenControl, WidgetControl, basemaps

# 1. Initialize map with satellite imagery
m = Map(center=WICHITA_FALLS, zoom=12, basemap=basemaps.Esri.WorldImagery)

# 2. Create layers (poly_layer starts hidden)
point_layer = GeoJSON(data=make_fc(points),   name="Points of Interest")
line_layer  = GeoJSON(data=make_fc(lines),    name="Routes")
poly_layer  = GeoJSON(data=make_fc(polygons), name="Park Boundary", visible=False)

# 3. Add layers and standard controls
m.add(point_layer)
m.add(line_layer)
m.add(poly_layer)
m.add(LayersControl(position="topright"))
m.add(ScaleControl(position="bottomleft"))
m.add(FullScreenControl(position="topleft"))

# 4. Manual MiniMap using WidgetControl
minimap_view = Map(
    center=m.center, zoom=m.zoom - 5, 
    zoom_control=False, attribution_control=False
)
minimap_view.layout.width, minimap_view.layout.height = '150px', '150px'
m.add(WidgetControl(widget=minimap_view, position='bottomright'))

m

Map(center=[33.9137, -98.4934], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'z…

---

## Check Your Understanding

Build a map centered on Wichita Falls at zoom 12 with:
- **two named GeoJSON layers** (points and polygons from `data/wichita_falls.geojson`)
- a `LayersControl` so both can be toggled
- a `ScaleControl` in the bottom-right corner

```python
# your answer here
```

In [22]:
import json
from ipyleaflet import Map, GeoJSON, LayersControl, ScaleControl

# 1. Load the data
with open("data/wichita_falls.geojson") as f:
    all_features = json.load(f)["features"]

# 2. Filter for Points and Polygons
points = [f for f in all_features if f["geometry"]["type"] == "Point"]
polygons = [f for f in all_features if f["geometry"]["type"] == "Polygon"]

# 3. Initialize the Map
m = Map(center=(33.9137, -98.4934), zoom=12)

# 4. Create named GeoJSON layers
# Note: make_fc is the helper function from your lesson to wrap the lists
def make_fc(features):
    return {"type": "FeatureCollection", "features": features}

point_layer = GeoJSON(data=make_fc(points), name="Points")
poly_layer = GeoJSON(data=make_fc(polygons), name="Polygons")

# 5. Add layers and the requested controls
m.add(point_layer)
m.add(poly_layer)

m.add(LayersControl(position="topright"))
m.add(ScaleControl(position="bottomright"))

m

Map(center=[33.9137, -98.4934], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'z…

## Next

In [03 — Attributes, Styling, and Filtering](../03-Attributes_Styling_Filtering/00-Properties.ipynb), we start reading feature properties and using them to style layers dynamically.